# Multimodal Mutation → Mechanism → Therapy demo

End-to-end walk-through of the multimodal pipeline on a single AMD AI Developer
Cloud GPU droplet running JupyterLab. **No Docker needed** — one `vllm serve`
process hosts the reasoning LLM, and three specialist models (ESM‑2,
BiomedCLIP/CLIP, Whisper) run in this kernel via HuggingFace `transformers`
on PyTorch‑ROCm.

Pipeline:
1. **vLLM** (text or VLM) on `http://127.0.0.1:8090/v1`
2. **ESM‑2** — zero‑shot variant‑effect score (sequence modality, second model)
3. **AlphaFold + Matplotlib 3‑D** — real backbone trace as an image (image modality)
4. **BiomedCLIP / CLIP** — optional histology / radiology image scoring
5. **Whisper** — optional clinician voice note transcription
6. **Reasoning chain** — fuses everything via the existing `reason()`

Run the cells top‑to‑bottom. Cells 4–6 are independently runnable so you can
validate each specialist before invoking the full pipeline.

## 1. Environment + GPU sanity check

Confirms ROCm is exposed to PyTorch, prints free VRAM, and shows what the
vLLM server is currently advertising.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))

import httpx

VLLM_URL = os.getenv("AI_BASE_URL", "http://127.0.0.1:8090/v1")

try:
    r = httpx.get(f"{VLLM_URL}/models", timeout=5.0)
    r.raise_for_status()
    served = [m['id'] for m in r.json().get('data', [])]
    print(f"vLLM at {VLLM_URL} — serving:", served)
except Exception as e:
    print(f"vLLM at {VLLM_URL} not reachable: {e}\n"
          "Start it in a JupyterLab terminal with `bash scripts/start_vllm.sh`.")

try:
    import torch
    print(f"torch={torch.__version__} cuda_available={torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"  device: {torch.cuda.get_device_name(0)}")
        free, total = torch.cuda.mem_get_info()
        print(f"  free VRAM: {free/2**30:.1f} / {total/2**30:.1f} GiB")
except Exception as e:
    print(f"torch not importable yet ({e}). Install with `pip install torch transformers` "
          "— use the ROCm wheel preinstalled in the AMD JupyterLab image.")

## 2. Pick a variant

In [ ]:
MUTATION = "BRAF V600E"
MODEL = os.getenv("AI_MODEL")  # text reasoner
VISION_MODEL = os.getenv("AI_VISION_MODEL")  # if you want the image attached
print(f"variant: {MUTATION}\nreasoning model: {MODEL or '(auto from .env)'}")

## 3. Specialist 1 — ESM‑2 zero‑shot variant effect (sequence modality)

Loads `facebook/esm2_t33_650M_UR50D` once into the kernel and computes
`delta_pll = log P(mut | context) - log P(WT | context)` after masking the
mutated residue. Strongly negative → disrupting; near zero → tolerated;
strongly positive → unusual / context‑dependent.

In [ ]:
from src.mutation import parse_mutation
from src import protein_lm

mq = parse_mutation(MUTATION)
esm = protein_lm.score_variant(mq)
esm

## 4. Specialist 2 — AlphaFold 3‑D backbone render (image modality)

Pulls the AlphaFold predicted structure for the gene, parses the Cα atoms,
and renders a backbone trace with the mutated residue highlighted as a red
sphere. This image is what the VLM sees on stage 2.

In [ ]:
import base64
from io import BytesIO
from PIL import Image

from src import structure as ds_struct

struct = ds_struct.fetch_structure(mq)
print({k: v for k, v in struct.items()
       if k not in ('domain_plot_png_b64', 'structure_3d_png_b64', 'features')})

for key, label in (("domain_plot_png_b64", "Domain map (2-D)"),
                   ("structure_3d_png_b64", "AlphaFold backbone (3-D)")):
    b64 = struct.get(key)
    if b64:
        print(f"\n{label}:")
        display(Image.open(BytesIO(base64.b64decode(b64))))
    else:
        print(f"{label}: not generated")

## 5. Specialist 3 — BiomedCLIP / CLIP image encoder (vision modality, optional)

If you have a histology / radiology image, drop it in `notebooks/` and set
`IMAGE_PATH` below. Skips cleanly if `open_clip_torch` and `transformers`
are both unavailable.

In [ ]:
from pathlib import Path
from src import imaging

IMAGE_PATH = None  # e.g. Path("sample_he.png")

if IMAGE_PATH and Path(IMAGE_PATH).exists():
    img_findings = imaging.encode_image(str(IMAGE_PATH))
    print(img_findings.get('summary') or img_findings)
else:
    print("No image attached — skip. Set IMAGE_PATH to a local file to enable.")
    img_findings = None

## 6. Specialist 4 — Whisper voice transcription (speech modality, optional)

Drop a `.wav` / `.mp3` in `notebooks/` and set `VOICE_PATH` to enable.

In [ ]:
from src import speech

VOICE_PATH = None  # e.g. "clinician_note.wav"

if VOICE_PATH and Path(VOICE_PATH).exists():
    sp = speech.transcribe(str(VOICE_PATH))
    print(sp)
else:
    print("No voice attached — skip. Set VOICE_PATH to enable.")
    sp = None

## 7. Full multimodal pipeline

`gather()` runs the public APIs, ESM‑2, AlphaFold render, optional image
scoring, and optional voice transcription **in parallel**. `reason()` then
calls the vLLM‑hosted reasoning model three times, attaching the structural
images on stage 2 if the model is vision‑capable.

In [ ]:
from src.evidence import gather
from src.reasoning import reason

image_bytes = Path(IMAGE_PATH).read_bytes() if IMAGE_PATH and Path(IMAGE_PATH).exists() else None
voice_bytes = Path(VOICE_PATH).read_bytes() if VOICE_PATH and Path(VOICE_PATH).exists() else None

mq, ev = gather(MUTATION, image=image_bytes, voice=voice_bytes)
print("Evidence blocks present:", [k for k, v in ev.to_dict().items()
                                    if isinstance(v, dict) and v])

In [ ]:
result = reason(mq, ev, model=VISION_MODEL or MODEL, verify=True, redact=True)

from IPython.display import Markdown, display
display(Markdown("### 1. Mutation Summary\n" + result.mutation_summary))
display(Markdown("### 2. Mechanism (multimodal stage)\n" + result.mechanism))
display(Markdown("### 3. Therapy\n" + result.therapy))

## 8. Grounding report

Hallucination metrics over all three stages, including the new
`[ESM2]` and `[BiomedCLIP]` citation tags.

In [ ]:
import json
print(json.dumps(result.grounding.get('aggregate', {}), indent=2))